In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.20 Identical Particles, Exchange Symmetry, and the Pauli Principle

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.20",
    title="Identical Particles, Exchange Symmetry, and the Pauli Principle",
    blurb="Why matter has structure. Two electrons are not just alike — they are "
    "identical, indistinguishable in a way that forces their joint state to be either "
    "symmetric or antisymmetric under exchange. From the antisymmetric case comes the "
    "Pauli principle and, with it, the periodic table, the solidity of solids, and the "
    "pressure that holds up a dying star. And from exchange alone, with no force to "
    "cause it, comes an energy that depends on spin — the root of magnetism and the "
    "chemical bond.",
    difficulty="advanced",
    estimate="170–210 min",
)

## Notebook overview

This is the notebook that explains why matter has the structure it does — the deepest "why is the world
this way" question the volume answers — and it grows from a single, almost trivial-sounding fact: two
electrons are not merely *similar*, they are **identical**. There is no measurement, even in principle,
that can tell one electron from another; they carry no labels. Classically we could still track them
("particle 1 started here, particle 2 there"), but quantum-mechanically we cannot, and quantum mechanics
enforces this indistinguishability with a postulate that turns out to build the periodic table.

The last notebook glimpsed the seed: two spin-$\tfrac12$'s combined into a *symmetric* triplet and an
*antisymmetric* singlet. Here that exchange symmetry is elevated from a feature of coupled states into a
**law**. Because swapping two identical particles can change nothing observable, the state must be an
eigenstate of the exchange operator, and there are only two possibilities: totally **symmetric** or
totally **antisymmetric**. Nature uses both — particles of integer spin are **bosons** (symmetric),
particles of half-integer spin are **fermions** (antisymmetric), the *spin–statistics connection* — and
electrons, being spin-$\tfrac12$, are fermions.

Three consequences follow, and we compute each. First, the **exchange hole**: the antisymmetric state
*vanishes* when two fermions coincide (and when they would occupy the same state), while the symmetric
state is *enhanced* there — a purely quantum correlation with no force behind it. Second, and genuinely
new, the **exchange interaction**: even with a Hamiltonian carrying *no* spin-dependent force, symmetric
and antisymmetric spatial states have different average separations and hence different interaction
energies — and because an electron pair's overall state must be antisymmetric, the *spatial* symmetry is
locked to the *spin* state (symmetric-spatial ↔ singlet, antisymmetric-spatial ↔ triplet, [§6.19](addition-angular-momenta.ipynb)). So the
energy depends on spin though the Hamiltonian does not: this is the origin of **Hund's rule**,
**ferromagnetism**, and the **covalent bond** — magnetism that is electrostatic in origin, filtered
through exchange symmetry. Third, the **Slater determinant** packages antisymmetry into one object that
vanishes automatically if two orbitals coincide — the **Pauli exclusion principle** — which forces
electrons into successive shells, giving the periodic table, the size and rigidity of matter, and the
degeneracy pressure that holds up white dwarfs.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the two-particle states via `numpy.outer` of single-particle
box orbitals ([§6.10](schrodinger-on-a-computer.ipynb)), `numpy.linalg.det` for the Slater determinant, and 2-D `numpy.trapezoid` sums for
$\langle(x_1-x_2)^2\rangle$ and $\langle V_{\text{int}}\rangle$.

> **Scope and conventions.** Single-particle states are the box eigenstates of [§6.10](schrodinger-on-a-computer.ipynb) ($\varphi_n(x)=
> \sqrt{2/L}\sin(n\pi x/L)$ on $[0,L]$, $L=1$). Two-particle wavefunctions are spatial only; the spin
> state is implied by the requirement that the *overall* fermion state be antisymmetric. **This notebook
> establishes the microscopic rule** (symmetrization, the exchange interaction, Slater/Pauli) **and its
> consequences for the structure of matter** (the periodic table, bonding, magnetism, stellar stability).
> It does **not** compute the quantum *statistics* — the Fermi–Dirac and Bose–Einstein *distributions*,
> quantum gases, blackbody radiation, Bose–Einstein condensation — which are the thermodynamics **built
> on** this rule, deferred to **Volume VII**. Indistinguishability is *why* quantum particles are counted
> differently from classical ones (the $1/N!$ and the correlations that Volume V's counting, [§5.1](../05-classical-stat-mech/counting.ipynb), could
> only anticipate); here is the microscopic foundation, and Volume VII builds the statistics on it. See
> Sakurai & Napolitano and Griffiths (identical particles, exchange, Slater determinants); and Notebooks
> [§6.19](addition-angular-momenta.ipynb) (singlet/triplet), [§6.8](bloch-sphere-entanglement.ipynb) (entanglement), [§6.17](hydrogen-atom.ipynb)/[§6.18](spin-magnetic.ipynb) (the $2n^2$ states), [§6.10](schrodinger-on-a-computer.ipynb) (the box orbitals),
> [§5.1](../05-classical-stat-mech/counting.ipynb) (counting).

## Theory in brief

### Indistinguishability and the two allowed symmetries

Exchanging two identical particles must leave every observable unchanged, so the exchange operator
$P_{12}$ (swap the particles) commutes with $H$ and every observable; since $P_{12}^2=I$, its eigenvalues
are $\pm1$, and physical states must be eigenstates:

```{math}
:label: eq-indistinguishable
P_{12}\Psi=\pm\Psi:\qquad \text{symmetric } (+1)\ \text{ or }\ \text{antisymmetric } (-1) .
```

### The symmetrization postulate and spin–statistics

Which sign a given species uses is not decided by {eq}`eq-indistinguishable`: nothing in
non-relativistic quantum mechanics ties the exchange eigenvalue to any other property of the particle.
Nature nevertheless makes a universal choice, taken here as a postulate:

```{math}
:label: eq-symmetrization
\text{integer spin}\Rightarrow\text{BOSONS (symmetric)},\qquad \text{half-integer spin}\Rightarrow\text{FERMIONS (antisymmetric)} ,
```

the **spin–statistics connection** — a theorem of relativistic quantum field theory (named here as a
horizon, not proved). Electrons, protons, neutrons are fermions; photons and many nuclei are bosons.

### Two-particle states and the exchange hole

From single-particle states $\varphi_a,\varphi_b$,

```{math}
:label: eq-two-particle
\Psi_{S/A}(x_1,x_2)=\frac{\varphi_a(x_1)\varphi_b(x_2)\pm\varphi_b(x_1)\varphi_a(x_2)}{\sqrt2} ,
```

with $+$ symmetric (bosons) and $-$ antisymmetric (fermions). $\Psi_A$ **vanishes** when $x_1=x_2$ (and
when $a=b$): fermions cannot coincide or share a state — the **exchange hole**. $\Psi_S$ is **enhanced**
there: bosons bunch. This is a correlation with *no force* behind it.

### The exchange interaction

Even with a spin-independent $H$, the average separation differs (Griffiths computes the three
expectation values in full):

```{math}
:label: eq-exchange
\langle(x_1-x_2)^2\rangle_S<\langle(x_1-x_2)^2\rangle_{\text{dist}}<\langle(x_1-x_2)^2\rangle_A ,
```

so with a repulsion $V(x_1,x_2)$ the antisymmetric state (particles farther) has lower energy. For
electrons the overall state is antisymmetric, tying spatial symmetry to spin (symmetric-spatial ↔
singlet, antisymmetric-spatial ↔ triplet, [§6.19](addition-angular-momenta.ipynb)), so the energy depends on spin though $H$ does not — the
**exchange interaction**, the origin of Hund's rule, ferromagnetism, and the covalent bond. *Magnetism is
electrostatic repulsion filtered through exchange symmetry.*

### The Slater determinant and Pauli exclusion

Antisymmetrizing $N$ particles by hand means summing the $N!$ permutations of the product
$\varphi_1\cdots\varphi_N$ with alternating signs: precisely the expansion of a determinant. The
$N$-fermion generalization of {eq}`eq-two-particle` is therefore

```{math}
:label: eq-slater
\Psi(x_1,\dots,x_N)=\frac{1}{\sqrt{N!}}\det\big[\varphi_i(x_j)\big] ,
```

automatically antisymmetric (swap two particles → swap two columns → sign flip), and **zero if two
orbitals coincide** (two equal rows) — the **Pauli exclusion principle**: no two identical fermions in
the same single-particle state.

### The architecture of matter

We now aim the exclusion principle at the atom. The one-electron states of [§6.17](hydrogen-atom.ipynb) carry the labels
$(n,l,m_l)$, spin ([§6.18](spin-magnetic.ipynb)) doubles them with $m_s$, and Pauli admits at most one electron per label, so

```{math}
:label: eq-matter
\text{fill } (n,l,m_l,m_s)\ \text{one electron each}\ \Rightarrow\ 2n^2\ \text{per shell}=2,8,18,32 ,
```

the periodic table. Exclusion also gives matter its **size and rigidity** (a degeneracy pressure —
electrons cannot all fall to the ground state) and holds up **white dwarfs** and **neutron stars**. And
indistinguishability is *why* quantum particles are counted differently from classical ones ([§5.1](../05-classical-stat-mech/counting.ipynb)) — the
microscopic foundation on which Volume VII's quantum statistics is built.

## Setup

The data are the series palette, the box itself ($L=1$) and its single-particle eigenstates
$\varphi_n(x)=\sqrt{2/L}\sin(n\pi x/L)$, which arrive as the given specimen from
[§6.10](schrodinger-on-a-computer.ipynb) — this notebook combines them, it does not derive them.
The instruments are the two-particle $(x_1,x_2)$ grid everything is sampled on and the
expectation-value integrator, a 2-D quadrature that measures the states rather than teaching
anything about them. The symmetrization itself is deliberately absent: you build the
two-particle state $\Psi_{S/A/D}$ in Exercise 1 and the Slater determinant in Exercise 5.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

# data: the series palette
ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# instrument: the two-particle configuration grid. The box [0, L] is discretized once and
# meshed into (x1, x2) so that every two-particle object in the notebook — the states you build
# in Exercise 1, the operators (x1−x2)² and V(x1,x2), the quadratures — lives on the same array.
# A sampling choice, not the lesson.
L = 1.0
N_GRID = 400
x = np.linspace(0, L, N_GRID)
X1, X2 = np.meshgrid(x, x, indexing="ij")


# data: the given single-particle states. The box eigenstates of §6.10 are the specimen this
# notebook combines into two-particle states; the formula is displayed above and there is
# nothing here to construct beyond transcribing it.
def box_orbital(n):
    """The n-th particle-in-a-box eigenstate φ_n(x)=√(2/L) sin(nπx/L) (the §6.10 box states), normalized."""
    f = np.sqrt(2 / L) * np.sin(n * np.pi * x / L)
    return f / np.sqrt(np.trapezoid(f**2, x))


# instrument: the measuring device. It reads an expectation value off a state by 2-D quadrature
# and is indifferent to which state it is handed — the physics lives in the states (Exercise 1)
# and in the operators the exercises build, not in the trapezoid rule.
def exchange_expectation(Psi, operator2d):
    r"""The expectation $\langle O\rangle=\iint|\Psi|^2 O(x_1,x_2)\,dx_1dx_2$ by 2-D ``numpy.trapezoid``."""
    integrand = np.abs(Psi) ** 2 * operator2d
    return np.trapezoid(np.trapezoid(integrand, x, axis=1), x)

## Exercise 1 — Exchange symmetry and the two allowed states

Because identical particles carry no labels, the joint state must be an eigenstate of the exchange
operator, and only two eigenvalues are available: $\Psi_S(x_2,x_1)=+\Psi_S(x_1,x_2)$ or
$\Psi_A(x_2,x_1)=-\Psi_A(x_1,x_2)$ {eq}`eq-indistinguishable`. From two single-particle orbitals
$\varphi_a,\varphi_b$ the two allowed combinations are the $\pm$ pair of {eq}`eq-two-particle`,
and spin–statistics {eq}`eq-symmetrization` assigns them: integer spin uses $\Psi_S$ (bosons),
half-integer spin uses $\Psi_A$ (fermions). There is no third possibility — only two exchange
symmetries exist.

On a grid, a product $\varphi_a(x_1)\varphi_b(x_2)$ is precisely the **outer product**
`numpy.outer(phi_a, phi_b)` of the two sampled orbitals, a 2-D array over $(x_1,x_2)$, and
exchanging the particles is transposing that array. The third, unsymmetrized combination — the
bare product $\Psi_D=\varphi_a(x_1)\varphi_b(x_2)$, which belongs to neither symmetry class — is
the *distinguishable* reference state that Exercises 3 and 4 measure the symmetric and
antisymmetric states against, so it is worth having from the same constructor. Each state is
normalized on the grid with a 2-D `numpy.trapezoid`.

1. Write `two_particle_state(phi_a, phi_b, symmetry)`, returning the normalized two-particle
   state on the $(x_1,x_2)$ grid: `'S'` the symmetric $\Psi_S$, `'A'` the antisymmetric
   $\Psi_A$, `'D'` the distinguishable product. **Write this one yourself** — the implementation
   is the lesson.
2. Take two single-particle box states $\varphi_a=\varphi_1$, $\varphi_b=\varphi_2$ and form
   $\Psi_S$ and $\Psi_A$.
3. Verify the symmetry under $x_1\leftrightarrow x_2$ (transposing the 2-D array).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    sym_ok and antisym_ok,
    "identical-particle states are symmetric (bosons) or antisymmetric (fermions) under exchange",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The exchange hole

Set $x_1=x_2$ in {eq}`eq-two-particle` and the antisymmetric numerator becomes
$\varphi_a(x)\varphi_b(x)-\varphi_b(x)\varphi_a(x)$, which is identically zero, while the
symmetric one doubles. Fermions therefore have *no* probability of being found at the same point
— the **exchange hole** — and bosons are enhanced there, **bunching**. On the 2-D grid that
coincidence line is just the diagonal of the array (`numpy.diag`). Nothing in the Hamiltonian
causes this: it is a statistical correlation from symmetry alone, the Pauli hole and bosonic
bunching before any force.

1. Evaluate $|\Psi_A|$ and $|\Psi_S|$ on the diagonal $x_1=x_2$, using the states you built in
   Exercise 1.
2. Confirm $\Psi_A=0$ there (the exchange hole) and $\Psi_S>0$ (bunching).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    diag_A.max(),
    0.0,
    "the antisymmetric state vanishes when the particles coincide — the exchange hole (Pauli, before any force)",
    atol=1e-6,
)

## Exercise 3 — The exchange effect on separation

The hole and the bunching of Exercise 2 are visible in a single number: the average squared
separation $\langle(x_1-x_2)^2\rangle$, which for the same two orbitals obeys
$\langle\cdot\rangle_S<\langle\cdot\rangle_{\text{dist}}<\langle\cdot\rangle_A$
{eq}`eq-exchange`. The distinguishable product $\Psi_D$ is the classical reference that the two
symmetrized states straddle, so the symmetric state behaves like an effective *attraction* and
the antisymmetric one like an effective *repulsion* — symmetry acts like a force, though there is
no force in $H$. The expectation values are 2-D quadratures over the grid,
$\langle O\rangle=\iint|\Psi|^2O\,dx_1dx_2$, which is what the `exchange_expectation` instrument
in the Setup performs.

1. Build the operator $(x_1-x_2)^2$ on the grid.
2. Build the distinguishable product $\Psi_D$ with the `two_particle_state` you wrote in
   Exercise 1, and compute $\langle(x_1-x_2)^2\rangle$ for $\Psi_S$, $\Psi_A$ and $\Psi_D$ with
   `exchange_expectation`.
3. Confirm the ordering symmetric $<$ distinguishable $<$ antisymmetric.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    sep_ok,
    "⟨(x₁−x₂)²⟩ is smallest for symmetric and largest for antisymmetric — exchange pulls symmetric states together and pushes antisymmetric apart",
)

## Exercise 4 — The exchange energy

Different average separations become different *energies* the moment an interaction is switched
on {eq}`eq-exchange`. A soft (regularized) Coulomb repulsion $V(x_1,x_2)=1/\sqrt{(x_1-x_2)^2+
\epsilon^2}$ serves: the $\epsilon$ keeps the integrand finite on the coincidence diagonal, where
the true $1/|x_1-x_2|$ would diverge. Since the antisymmetric state keeps the particles farther
apart, it pays *less* repulsion, and the difference $\langle V\rangle_S-\langle V\rangle_A$ is the
**exchange energy**.

For electrons the point is what this couples to. The *overall* state must be antisymmetric, so
the spatial symmetry is locked to the spin state — symmetric-spatial ↔ singlet,
antisymmetric-spatial ↔ triplet ([§6.19](addition-angular-momenta.ipynb)) — and the energy therefore depends on the spin
state although $V$ has no spin dependence at all. That is the exchange interaction, and it is the
origin of Hund's rule, ferromagnetism, and the covalent bond: a spin-dependent energy from a
spin-independent Hamiltonian.

1. Build the soft repulsion $V(x_1,x_2)$ on the grid.
2. Compute $\langle V\rangle$ for $\Psi_S$ and $\Psi_A$ with `exchange_expectation`.
3. Show the antisymmetric state (particles farther) has *lower* interaction energy, and read off
   the exchange energy.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    exchange_ok,
    "⟨V_int⟩ differs between Ψ_S and Ψ_A (the exchange energy) — a spin-dependent energy with no spin-dependent force",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — The Slater determinant and Pauli exclusion

Antisymmetrizing $N$ particles by hand means summing all $N!$ permutations of the product
$\varphi_1\cdots\varphi_N$ with alternating signs, and that sum is exactly the expansion of a
determinant {eq}`eq-slater`. Everything follows from how the matrix is indexed: put orbital $i$
in row $i$ and particle $j$ in column $j$, $M_{ij}=\varphi_i(x_j)$. Then *swapping two particles*
swaps two columns, and a determinant changes sign — the state is antisymmetric by construction;
and *giving two particles the same orbital* repeats a row, and a determinant with two equal rows
is zero — no two identical fermions may occupy the same single-particle state, the **Pauli
exclusion principle**. Antisymmetry and exclusion come packaged in one object.

The determinant is evaluated at a handful of particle *positions* rather than on the $(x_1,x_2)$
grid, so the orbitals enter as callables of a scalar position: the small `box` factory below
returns $\varphi_n$ in that form.

1. Write `slater_determinant(orbitals, points)`, forming $M_{ij}=\varphi_i(x_j)$ from a list of
   orbital callables and a list of positions and returning `numpy.linalg.det(M)`. **Write this
   one yourself** — the implementation is the lesson.
2. Evaluate it for three distinct orbitals $\varphi_1,\varphi_2,\varphi_3$ at three positions.
3. Verify that swapping two particles (two columns) flips the sign.
4. Show that with two *identical* orbitals (two equal rows) the determinant is zero.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    det_repeated,
    0.0,
    "the Slater determinant vanishes when two orbitals coincide — the Pauli exclusion principle",
    atol=1e-9,
)

## Exercise 6 — Building the periodic table *(student)*

Aim exclusion at the atom. The one-electron states of [§6.17](hydrogen-atom.ipynb) carry labels $(n,l,m_l)$, spin
([§6.18](spin-magnetic.ipynb)) doubles them with $m_s$, and Pauli admits at most one electron per label, so shell $n$
holds $\sum_{l=0}^{n-1}2(2l+1)=2n^2$ electrons {eq}`eq-matter`. That is the entire architecture of
the periodic table from one rule: the capacities $2,8,18,32$, the chemistry set by the handful of
electrons in the outermost partly-filled shell, and the reason electrons do not all collapse into
$n=1$ — which is why matter has a size at all.

1. List the states $(n,l,m_l,m_s)$ of each shell $n$ and place one electron per state (Pauli).
2. Count the electrons at each shell closing and confirm the $2n^2$ capacities.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    capacities_ok,
    "filling states one-per-(n,l,m_l,m_s) gives the 2n²=2,8,18,32 shell structure — the Pauli principle builds the periodic table",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Bosons versus fermions in a trap *(student)*

The same trap, the same two particles, two utterly different ground states — decided by nothing
but the sign under exchange {eq}`eq-symmetrization`, {eq}`eq-two-particle`. Two **bosons** may
both occupy the ground orbital $\varphi_1$, since the symmetric combination survives when
$a=b$; two **fermions** may not, because the antisymmetric combination vanishes identically for
$a=b$, so the second is pushed up to $\varphi_2$. The box energies are $E_n=n^2$ in units of
$\hbar^2\pi^2/2mL^2$. The energy the fermions pay for this is the Pauli "pressure" that gives
matter its size, while the bosonic tendency to pile into a single state anticipates
Bose–Einstein condensation (the thermodynamics is Volume VII).

1. Compute the total ground-state energy of two bosons and of two fermions in the box.
2. Compare them and read off the fermionic cost.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    trap_ok,
    "two bosons share the ground orbital while two fermions must fill two orbitals (Pauli) — exchange symmetry dictates how identical particles occupy states",
)

## Exercise 8 — One rule, and the shape of the world *(synthesis)*

The whole of this notebook grew from a single impossibility — that two identical particles cannot be told
apart — which forced their joint state to be symmetric or antisymmetric and split the particles of the
world into bosons and fermions. From antisymmetry came the **Pauli principle**, and from it the periodic
table, the solidity and size of matter, and the degeneracy pressure inside a white dwarf. From
**exchange** came an energy that depends on spin with no magnetic force behind it, and with it magnetism
and the chemical bond.

There is no new computation here: the structure of matter is the result. **Movement IV
is complete** — we have the electron's spin ([§6.18](spin-magnetic.ipynb)), the coupling of angular momenta ([§6.19](addition-angular-momenta.ipynb)), and now the
law of identical particles (§6.20): everything needed to describe real atoms. We should also be clear
about the boundary we have drawn. This notebook gave the *microscopic rule* — indistinguishability,
symmetrization, the exchange interaction, Pauli exclusion — and its consequences for the *structure* of
matter. Building the *thermodynamics* on it — the Fermi–Dirac and Bose–Einstein distributions, quantum
gases, blackbody radiation, Bose–Einstein condensation — is the work of **Volume VII**, which stands on
exactly this foundation (and completes the indistinguishable-counting that Volume V's [§5.1](../05-classical-stat-mech/counting.ipynb) could only
anticipate). The final movement of *this* volume turns instead to a different practical fact: almost no
real system is exactly solvable, so we need **approximation methods** — perturbation theory, the
variational method, the semiclassical limit — beginning with the fine structure that spin–orbit coupling
adds to the hydrogen levels ([§6.21](perturbation-fine-structure.ipynb)).

It is hard to overstate how much rides on a minus sign. Symmetric or antisymmetric — that single choice
under exchange decides whether particles pile into one state or refuse to share it, and the refusal is
why you are not falling through your chair. The stability of matter is, in the end, a theorem about the
sign of a wavefunction.

## Notebook summary

Identical particles and the Pauli principle — the close of Movement IV, and why matter has structure.

- **Indistinguishability** {eq}`eq-indistinguishable`, {eq}`eq-symmetrization`: states must be symmetric
  (**bosons**, integer spin) or antisymmetric (**fermions**, half-integer spin) — the spin–statistics
  connection (named, not proved).
- **The exchange hole** {eq}`eq-two-particle`: $\Psi_A$ vanishes when fermions coincide, $\Psi_S$ is
  enhanced when bosons do — correlations from symmetry, with no force (`numpy.outer`).
- **The exchange interaction** {eq}`eq-exchange`: $\langle(x_1-x_2)^2\rangle_S<\langle\cdot\rangle_A$, so
  a spin-independent repulsion gives a spin-dependent energy (spatial ↔ spin symmetry, [§6.19](addition-angular-momenta.ipynb)) — Hund's
  rule, magnetism, the covalent bond.
- **The Slater determinant** {eq}`eq-slater`: $\det[\varphi_i(x_j)]$ (`numpy.linalg.det`) is
  antisymmetric and vanishes for a repeated orbital — the **Pauli exclusion principle**.
- **The architecture of matter** {eq}`eq-matter`: filling $(n,l,m_l,m_s)$ by exclusion gives $2n^2=2,8,
  18,32$ — the periodic table, the size of atoms, the pressure in white dwarfs.

One minus sign under exchange builds the periodic table and holds up your chair. Movement IV is complete;
the quantum *statistics* built on this rule is Volume VII.

## Outlook

- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))** — the opening of Movement V: spin–orbit and relativistic corrections to the
  hydrogen levels, by perturbation theory.
- **Quantum statistics (Volume VII)**: the Fermi–Dirac and Bose–Einstein distributions, quantum gases,
  blackbody radiation, Bose–Einstein condensation — the thermodynamics built on this notebook's rule.
- **The many-electron atom and quantum chemistry**: Hartree–Fock, the aufbau/Madelung filling, screening
  (horizons).
- **Degeneracy pressure and the stability of stars**: white dwarfs and neutron stars (a horizon).
- **Cross-reference** [§6.19](addition-angular-momenta.ipynb) (singlet/triplet), [§6.8](bloch-sphere-entanglement.ipynb) (entanglement), [§6.17](hydrogen-atom.ipynb)/[§6.18](spin-magnetic.ipynb) (the $2n^2$ states), [§5.1](../05-classical-stat-mech/counting.ipynb)
  (counting — the classical anticipation), and forward to [§6.21](perturbation-fine-structure.ipynb), Volume VII.

In [ ]:
from ecp.style import footer

footer()